In [2]:
import requests
import time
import pandas as pd
from bs4 import BeautifulSoup
from textblob import TextBlob
import json

# ----------------------
# PARAMETERS
# ----------------------
STACK_KEYWORDS = "stress"
STACK_SITE = "stackoverflow"
STACK_PAGE_SIZE = 50
STACK_MAX_PAGES = 2

GITHUB_KEYWORDS = "stress"
GITHUB_PER_PAGE = 50
GITHUB_TOKEN = "your_personal_access_token_here"

NUM_POSTS = 100
MAX_COMMENTS = 10

# ----------------------
# UTILITIES
# ----------------------
def clean_html(raw_html):
    if not raw_html:
        return ""
    soup = BeautifulSoup(raw_html, "html.parser")
    return soup.get_text().strip()

def sentiment_score(text):
    if not text:
        return 0
    return TextBlob(text).sentiment.polarity

# ----------------------
# STACKOVERFLOW DATA
# ----------------------
def fetch_stackoverflow_posts():
    posts = []
    print("Fetching StackOverflow posts...")

    for page in range(1, STACK_MAX_PAGES + 1):

        search_url = "https://api.stackexchange.com/2.3/search/advanced"
        params = {
            "q": STACK_KEYWORDS,
            "site": STACK_SITE,
            "pagesize": STACK_PAGE_SIZE,
            "page": page,
            "order": "desc",
            "sort": "votes"
        }

        search_res = requests.get(search_url, params=params).json()
        items = search_res.get("items", [])
        print(f"Page {page}: {len(items)} questions")

        for q in items:
            qid = q["question_id"]

            # Fetch question body
            q_url = f"https://api.stackexchange.com/2.3/questions/{qid}"
            q_res = requests.get(q_url, params={"site": STACK_SITE, "filter": "withbody"}).json()

            if not q_res.get("items"):
                continue

            question = q_res["items"][0]

            # Fetch comments
            c_url = f"https://api.stackexchange.com/2.3/questions/{qid}/comments"
            c_res = requests.get(c_url, params={"site": STACK_SITE, "sort": "votes", "pagesize": MAX_COMMENTS}).json()

            comments = [
                {
                    "comment_id": c.get("comment_id", "unknown"),
                    "author": c.get("owner", {}).get("display_name", "unknown"),
                    "body": clean_html(c.get("body", "")),
                    "score": c.get("score", 0)
                }
                for c in c_res.get("items", [])
            ]

            posts.append({
                "platform": "stackoverflow",
                "post_id": str(qid),
                "title": question["title"],
                "selftext": clean_html(question["body"]),
                "top_10_comments": comments,
                "sentiment": sentiment_score(clean_html(question["body"]))
            })

            time.sleep(0.2)
            if len(posts) >= NUM_POSTS:
                return posts

    return posts

# ----------------------
# GITHUB DATA
# ----------------------
def fetch_github_issues():
    headers = {"Accept": "application/vnd.github+json"}
    if GITHUB_TOKEN:
        headers["Authorization"] = f"token {GITHUB_TOKEN}"

    posts = []
    print("Fetching GitHub issues...")

    search_url = "https://api.github.com/search/issues"
    params = {
        "q": f"{GITHUB_KEYWORDS} in:title,body is:issue",
        "per_page": GITHUB_PER_PAGE
    }

    search_res = requests.get(search_url, headers=headers, params=params).json()
    items = search_res.get("items", [])
    print(f"GitHub issues found: {len(items)}")

    for item in items[:NUM_POSTS]:

        # Extract repo owner and repo name
        repo_url = item["repository_url"]
        _, _, _, _, owner, repo = repo_url.split("/")

        issue_number = item["number"]

        # Fetch comments
        comments_url = f"https://api.github.com/repos/{owner}/{repo}/issues/{issue_number}/comments"
        comments_res = requests.get(comments_url, headers=headers, params={"per_page": MAX_COMMENTS}).json()

        comments = [
            {
                "comment_id": c.get("id", "unknown"),
                "author": c.get("user", {}).get("login", "unknown"),
                "body": clean_html(c.get("body", "")),
            }
            for c in comments_res
        ]

        posts.append({
            "platform": "github",
            "post_id": str(item["id"]),
            "title": item["title"],
            "selftext": clean_html(item["body"]) if item["body"] else "",
            "top_10_comments": comments,
            "sentiment": sentiment_score(item["body"] or "")
        })

    return posts

# ----------------------
# RUN COLLECTION
# ----------------------
if __name__ == "__main__":
    stack_posts = fetch_stackoverflow_posts()
    github_posts = fetch_github_issues()

    combined_posts = stack_posts + github_posts
    print(f"Total posts collected: {len(combined_posts)}")

    # Save JSON
    with open("developer_stress_posts.json", "w", encoding="utf-8") as f:
        json.dump(combined_posts, f, ensure_ascii=False, indent=4)

    # Save CSV
    flat_data = []
    for post in combined_posts:
        for comment in post["top_10_comments"]:
            flat_data.append({
                "title": post["title"],
                "selftext": post["selftext"],
                "comment_author": comment["author"],
                "comment_body": comment["body"],
            })

    df = pd.DataFrame(flat_data)
    df.to_csv("developer_stress_posts.csv", index=False, encoding="utf-8")

    print("Data collection complete! JSON and CSV saved.")


Fetching StackOverflow posts...
Page 1: 50 questions
Page 2: 50 questions
Fetching GitHub issues...
GitHub issues found: 0
Total posts collected: 100
Data collection complete! JSON and CSV saved.
